In [6]:
import pandas as pd
import numpy as np

from src.data import load_train, load_test

# Load datasets
train = load_train()
test = load_test()

print("Training shape:", train.shape)
print("Testing shape:", test.shape)

Training shape: (175341, 45)
Testing shape: (82332, 45)


In [7]:
# Compare training and testing columns

print("Training columns:", len(train.columns))
print("Testing columns:", len(test.columns))

print("\nColumns missing from testing:")
print(set(train.columns) - set(test.columns))

print("\nExtra columns in testing:")
print(set(test.columns) - set(train.columns))

print("\nAre columns in the same order?")
print(list(train.columns) == list(test.columns))

Training columns: 45
Testing columns: 45

Columns missing from testing:
set()

Extra columns in testing:
set()

Are columns in the same order?
True


In [8]:
print("Training labels:")
print(train["label"].value_counts().sort_index())

print("\nTesting labels:")
print(test["label"].value_counts().sort_index())

print("\nUnique training labels:")
print(train["label"].unique())

print("\nUnique testing labels:")
print(test["label"].unique())

Training labels:
label
0     56000
1    119341
Name: count, dtype: int64

Testing labels:
label
0    37000
1    45332
Name: count, dtype: int64

Unique training labels:
[0 1]

Unique testing labels:
[0 1]


In [9]:
# Select numerical columns
train_numeric = train.select_dtypes(include="number")
test_numeric = test.select_dtypes(include="number")

# Count infinite values
print("Infinite values in training:")
print(np.isinf(train_numeric).sum().sum())

print("\nInfinite values in testing:")
print(np.isinf(test_numeric).sum().sum())

# Count NaN values in numerical columns
print("\nNaN values in training:")
print(train_numeric.isna().sum().sum())

print("\nNaN values in testing:")
print(test_numeric.isna().sum().sum())

Infinite values in training:
0

Infinite values in testing:
0

NaN values in training:
0

NaN values in testing:
0


In [10]:
categorical_columns = ["proto", "service", "state"]

for col in categorical_columns:
    print(f"\n--- {col} ---")

    train_values = set(train[col].dropna().unique())
    test_values = set(test[col].dropna().unique())

    print("Unique values in training:", len(train_values))
    print("Unique values in testing:", len(test_values))

    print("Values only in testing:")
    print(test_values - train_values)

    print("Missing values in training:", train[col].isna().sum())
    print("Missing values in testing:", test[col].isna().sum())


--- proto ---
Unique values in training: 133
Unique values in testing: 131
Values only in testing:
set()
Missing values in training: 0
Missing values in testing: 0

--- service ---
Unique values in training: 13
Unique values in testing: 13
Values only in testing:
set()
Missing values in training: 0
Missing values in testing: 0

--- state ---
Unique values in training: 9
Unique values in testing: 7
Values only in testing:
{'CLO', 'ACC'}
Missing values in training: 0
Missing values in testing: 0


In [11]:
print("Training duplicate IDs:", train["id"].duplicated().sum())
print("Testing duplicate IDs:", test["id"].duplicated().sum())

overlapping_ids = set(train["id"]) & set(test["id"])

print("IDs appearing in both datasets:", len(overlapping_ids))

Training duplicate IDs: 0
Testing duplicate IDs: 0
IDs appearing in both datasets: 82332


In [12]:
print("Training ID range:")
print("Minimum:", train["id"].min())
print("Maximum:", train["id"].max())

print("\nTesting ID range:")
print("Minimum:", test["id"].min())
print("Maximum:", test["id"].max())

print("\nFirst 10 training IDs:")
print(train["id"].head(10).tolist())

print("\nFirst 10 testing IDs:")
print(test["id"].head(10).tolist())

Training ID range:
Minimum: 1
Maximum: 175341

Testing ID range:
Minimum: 1
Maximum: 82332

First 10 training IDs:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

First 10 testing IDs:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [13]:
# Columns that must not be used as model features
excluded_columns = ["id", "attack_cat", "label"]

# Create the feature column list
feature_columns = [
    col for col in train.columns
    if col not in excluded_columns
]

print("Excluded columns:", excluded_columns)
print("\nNumber of input features:", len(feature_columns))
print("\nInput features:")
print(feature_columns)

print("\nIs label included in features?", "label" in feature_columns)
print("Is attack_cat included in features?", "attack_cat" in feature_columns)
print("Is id included in features?", "id" in feature_columns)

Excluded columns: ['id', 'attack_cat', 'label']

Number of input features: 42

Input features:
['dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports']

Is label included in features? False
Is attack_cat included in features? False
Is id included in features? False
